In [1]:
import langchain_core
from langchain_classic.agents import AgentType, initialize_agent, load_tools
from langchain_classic.chains.sequential import SequentialChain
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from config_loader import load_config_as_dict

In [3]:
llm_model: ChatOllama = None
llm_chain: LLMChain = None

def llm_initializer():
    global llm_model
    if llm_model:
        print("(*) LLM Model Already Initialized.")
        return
    llm_config = load_config_as_dict()['llm']
    llm_config['model'] = 'qwen3:8b'  # Didn't change the config.yaml file so changed here
    llm_model = ChatOllama(
        model=llm_config['model'],
        temperature=llm_config['temperature'],
        num_predict=llm_config['max_tokens'],  # Ollama's equivalent of Groq's max_tokens
        num_ctx=llm_config['num_ctx'],
        reasoning=llm_config['reasoning'],
        keep_alive=llm_config['keep_alive'],
    )
    print("(*) LLM Model Initialized.")


def initialize_llm_chain():
    global llm_chain, llm_model
    if llm_chain:
        print("(*) LLM Chain Already Initialized.")
    query = """I want to open a restaurant for {cuisine} food. Suggest a delicate yet fancy name for this.
    ** SHARE JUST THE BEST NAME**, only 1 name and nothing else."""
    prompt_template_name = PromptTemplate(
        input_variables=['cuisine'],
        template=query
    )
    llm_chain = LLMChain(llm=llm_model, prompt=prompt_template_name, output_key='restaurant_name')
    print("(*) LLM Chain Initialized.")

def initialize_llm_chain_with_memory(memory):
    global llm_chain_with_memory, llm_model
    print("(*) Initializing LLM Chain With Memory.")
    query = """I want to open a restaurant for {cuisine} food. Suggest a delicate yet fancy name for this.
    ** SHARE JUST THE BEST NAME**, only 1 name and nothing else."""
    prompt_template_name = PromptTemplate(
        input_variables=['cuisine'],
        template=query
    )
    llm_chain_with_memory = LLMChain(llm=llm_model, prompt=prompt_template_name, output_key='restaurant_name', memory=memory)
    print("(*) LLM Chain Initialized With Memory.")

In [4]:
def run(cuisine):
    global llm_chain, llm_model
    if not llm_model:
        llm_initializer()

    if not llm_chain:
        initialize_llm_chain()
    return llm_chain.invoke(cuisine)

name = run('Indian')
name

(*) LLM Model Initialized.
(*) LLM Chain Initialized.


C:\Users\ShubhanshuJha\AppData\Local\Temp\ipykernel_26576\308437920.py:32: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm=llm_model, prompt=prompt_template_name, output_key='restaurant_name')


{'cuisine': 'Indian', 'restaurant_name': 'Aurora Bazaar'}

In [5]:
name = run('Maxican')
name

{'cuisine': 'Maxican', 'restaurant_name': 'Cocina de Noche'}

In [6]:
type(llm_chain)

langchain_classic.chains.llm.LLMChain

In [7]:
type(llm_chain.memory)  ### Should show NoneType, means no memory

NoneType

### Adding Memory to the LLM

In [8]:
from langchain_classic.memory import ConversationBufferMemory

llm_chain_with_memory: LLMChain = None

def run(cuisine):
    global llm_chain_with_memory, llm_model
    if not llm_model:
        llm_initializer()
    if not llm_chain_with_memory:
        memory = ConversationBufferMemory()
        initialize_llm_chain_with_memory(memory=memory)
    return llm_chain_with_memory.invoke(cuisine)

name = run('Indian')
name

C:\Users\ShubhanshuJha\AppData\Local\Temp\ipykernel_26576\3105472441.py:10: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory()


(*) Initializing LLM Chain With Memory.
(*) LLM Chain Initialized With Memory.


{'cuisine': 'Indian', 'history': '', 'restaurant_name': 'Aurora Bazaar'}

In [13]:
name = run('Chinese')
name

{'cuisine': 'Chinese',
 'history': 'Human: Indian\nAI: Aurora Bazaar',
 'restaurant_name': '翠宴轩'}

In [14]:
print(type(llm_chain_with_memory.memory))
llm_chain_with_memory.memory

<class 'langchain_classic.memory.buffer.ConversationBufferMemory'>


ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content='Indian', additional_kwargs={}, response_metadata={}), AIMessage(content='Aurora Bazaar', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Chinese', additional_kwargs={}, response_metadata={}), AIMessage(content='翠宴轩', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]))

In [15]:
print(llm_chain_with_memory.memory.buffer)

Human: Indian
AI: Aurora Bazaar
Human: Chinese
AI: 翠宴轩


##### NOTE: ConversationBufferMemory will keep growing the memory buffer so not ideal where short term memory is required.

In [19]:
### ConversationChain sends whole convo history to the LLM so could increase token cost
from langchain_classic.chains import ConversationChain

convo_chain = ConversationChain(llm=llm_model)
convo_chain.prompt

C:\Users\ShubhanshuJha\AppData\Local\Temp\ipykernel_26576\1020334658.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  convo_chain = ConversationChain(llm=llm_model)


PromptTemplate(input_variables=['history', 'input'], input_types={}, partial_variables={}, template='The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.\n\nCurrent conversation:\n{history}\nHuman: {input}\nAI:')

In [20]:
print(convo_chain.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


In [21]:
query = "Who won the first cricket world cup?"
convo_chain.run(query)

C:\Users\ShubhanshuJha\AppData\Local\Temp\ipykernel_26576\2106311832.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  convo_chain.run(query)


'The first Cricket World Cup was won by the West Indies. They defeated the host nation, Australia, in the final match held in 1975 in England. The West Indies team, known for their aggressive style of play, was led by the legendary player Clive Lloyd. This victory marked the beginning of a successful era for West Indies cricket, and they went on to win the tournament again in 1979.'

In [22]:
query = "If x belongs to {2, 4, 6}, then what is x^2?"
convo_chain.invoke(query)

{'input': 'If x belongs to {2, 4, 6}, then what is x^2?',
 'history': 'Human: Who won the first cricket world cup?\nAI: The first Cricket World Cup was won by the West Indies. They defeated the host nation, Australia, in the final match held in 1975 in England. The West Indies team, known for their aggressive style of play, was led by the legendary player Clive Lloyd. This victory marked the beginning of a successful era for West Indies cricket, and they went on to win the tournament again in 1979.',
 'response': 'If $ x $ belongs to the set $ \\{2, 4, 6\\} $, then we can calculate $ x^2 $ for each value of $ x $:\n\n- When $ x = 2 $, $ x^2 = 2^2 = 4 $\n- When $ x = 4 $, $ x^2 = 4^2 = 16 $\n- When $ x = 6 $, $ x^2 = 6^2 = 36 $\n\nSo, the set of values for $ x^2 $ is $ \\{4, 16, 36\\} $.'}

In [23]:
query = "Who was the captain of the winning team?"
convo_chain.run(query)

"The captain of the West Indies team that won the first Cricket World Cup in 1975 was Clive Lloyd. He was a legendary cricketer known for his leadership, aggressive batting style, and pivotal role in the team's success during that era. Lloyd also captained the team to another World Cup victory in 1979."

In [24]:
print(convo_chain.memory.buffer)

Human: Who won the first cricket world cup?
AI: The first Cricket World Cup was won by the West Indies. They defeated the host nation, Australia, in the final match held in 1975 in England. The West Indies team, known for their aggressive style of play, was led by the legendary player Clive Lloyd. This victory marked the beginning of a successful era for West Indies cricket, and they went on to win the tournament again in 1979.
Human: If x belongs to {2, 4, 6}, then what is x^2?
AI: If $ x $ belongs to the set $ \{2, 4, 6\} $, then we can calculate $ x^2 $ for each value of $ x $:

- When $ x = 2 $, $ x^2 = 2^2 = 4 $
- When $ x = 4 $, $ x^2 = 4^2 = 16 $
- When $ x = 6 $, $ x^2 = 6^2 = 36 $

So, the set of values for $ x^2 $ is $ \{4, 16, 36\} $.
Human: Who was the captain of the winning team?
AI: The captain of the West Indies team that won the first Cricket World Cup in 1975 was Clive Lloyd. He was a legendary cricketer known for his leadership, aggressive batting style, and pivotal r

#### ConversationChain alone is not good for limited window context. Use ConversationBufferWindowMemory with required window configuration.

In [26]:
from langchain_classic.memory import ConversationBufferWindowMemory

lim_llm_memory = ConversationBufferWindowMemory(k=1)
convo_chain = ConversationChain(llm=llm_model, memory=lim_llm_memory)
print(convo_chain.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


In [27]:
query = "Who won the first cricket world cup?"
convo_chain.run(query)

'The first Cricket World Cup was won by the West Indies. They defeated Australia in the final, which took place in 1975 in England. The West Indies team, known for their aggressive style of play, was led by captain Clive Lloyd and included legendary players like Joel Garner, Michael Holding, and Vivian Richards. This victory marked a significant moment in cricket history, as it was the first time the tournament was held, and the West Indies showcased their dominance on the global stage.'

In [28]:
query = "What is 5*5?"
convo_chain.run(query)

'5 multiplied by 5 equals 25.'

In [29]:
query = "Who was the captain of the winning team?"
convo_chain.run(query)

"I don't have enough information to answer that question. Could you please provide more context about which team or event you're referring to?"

In [30]:
### The LLM now stores only 1 conversation as history
print(convo_chain.memory.buffer)

Human: Who was the captain of the winning team?
AI: I don't have enough information to answer that question. Could you please provide more context about which team or event you're referring to?


___
___